In [1]:
import ROOT
import numpy
import pandas

The data file was converted from the original `fullgeoanatruth-vd1x8x14-1000.root` with:
```
InputFile="fullgeoanatruth-vd1x8x14-100.root"
time ./convertLabelToCode.exe "$InputFile" "fullgeoanatruth/FullGeoAnaTruth" label "${InputFile%.root}_new.root"
```
(it took roughly 2 seconds for a 100-event file).

In [2]:
InputFilePath = "/Users/yuntse/data/lartpc_rd/gampix/gen/dune/fullgeoanatruth-vd1x8x14-1000_code.root"
InputTreeName = "FullGeoAnaTruth"

In [3]:
%%time
rootdf = ROOT.RDataFrame(InputTreeName, InputFilePath)
rootdf.Describe()

CPU times: user 193 ms, sys: 33 ms, total: 226 ms
Wall time: 425 ms


Dataframe from TChain FullGeoAnaTruth in file /Users/yuntse/data/lartpc_rd/gampix/gen/dune/fullgeoanatruth-vd1x8x14-1000_code.root

Property                Value
--------                -----
Columns in total           15
Columns from defines        0
Event loops run             0
Processing slots            1

Column          Type            Origin
------          ----            ------
Ekin            Double_t        Dataset
Etot            Double_t        Dataset
event           Int_t           Dataset
labelcode       UInt_t          Dataset
mass            Double_t        Dataset
pdg_code        Int_t           Dataset
px              Double_t        Dataset
py              Double_t        Dataset
pz              Double_t        Dataset
run             Int_t           Dataset
subrun          Int_t           Dataset
t               Double_t        Dataset
x               Double_t        Dataset
y               Double_t        Dataset
z               Double_t        Dataset

Loading the tree in Pandas took 15" for 100 events.

In [4]:
%%time
df = pandas.DataFrame(rootdf.AsNumpy())
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132301297 entries, 0 to 132301296
Data columns (total 15 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Ekin       float64
 1   Etot       float64
 2   event      int32  
 3   labelcode  uint32 
 4   mass       float64
 5   pdg_code   int32  
 6   px         float64
 7   py         float64
 8   pz         float64
 9   run        int32  
 10  subrun     int32  
 11  t          float64
 12  x          float64
 13  y          float64
 14  z          float64
dtypes: float64(10), int32(4), uint32(1)
memory usage: 12.3 GB
CPU times: user 1min 1s, sys: 15.6 s, total: 1min 16s
Wall time: 1min 46s


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132301297 entries, 0 to 132301296
Data columns (total 15 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Ekin       float64
 1   Etot       float64
 2   event      int32  
 3   labelcode  uint32 
 4   mass       float64
 5   pdg_code   int32  
 6   px         float64
 7   py         float64
 8   pz         float64
 9   run        int32  
 10  subrun     int32  
 11  t          float64
 12  x          float64
 13  y          float64
 14  z          float64
dtypes: float64(10), int32(4), uint32(1)
memory usage: 12.3 GB


We read the label index and store here in three different formats for convenience:
 * `labelToCode`: Python dictionary with the original `label` as key and the code as value
 * `codeTolabel`: Python dictionary with the code as key and the original `label` as value
 * `labelDF`    : a Pandas dataframe with columns `labelCode` and `label`

In [6]:
try:
    LabelTreeName = InputTreeName + "_LabelMap"
    FInput = ROOT.TFile(InputFilePath, "READ")
    LabelTree = FInput.Get(InputTreeName + "_LabelMap")
    if not LabelTree: raise RuntimeError(f"Can't read tree '{LabelTreeName}' in '{InputFilePath}.")
except:
    try: FInput.Close()
    except AttributeError: pass
    raise
labelToCode = {}
codeToLabel = {}
for row in LabelTree:
    label = str(row.label)
    code = row.labelCode
    labelToCode[label] = code
    codeToLabel[code] = label
# for
labelDF = pandas.DataFrame(dict(labelCode=codeToLabel.keys(), label=codeToLabel.values()))
labelDF

,labelCode,label
0,3397053793,CryostatNGammasAtLAr
1,3364292438,CavernNGammasAtLAr
2,544797560,CavernwallNeutronsAtLAr
3,655153075,Rn222ChainGenInPDS
4,3304492435,foamGammasAtLAr
5,3641398812,CavernwallGammasAtLAr
6,871716743,Th232ChainGenInAnode
7,597414879,U238ChainGenInAnode
8,722243711,K40GenInAnode
9,3162635185,Rn222ChainFromBi210GenInUpperMesh
